# 02a Retrain Best EMA Checkpoints

Run this notebook on Colab A100 High-RAM after Notebook 00 bootstrap has installed the ECG-RAMBA runtime. It retrains the five Chapman folds and writes explicit EMA/raw checkpoint variants to a versioned Drive model run directory, not the historical `Drive/ECG-Ramba/model` directory. Do not continue to Notebook 02 OOF until all five `fold*_best_ema.pt` files pass the verification cell and the current model-run pointer is written.


## Setup Repository And Drive Paths


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = 'main'
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
MODEL_RUNS_DIR = DRIVE_ROOT / 'model_runs'
MODEL_RUN_ID = os.environ.get('ECG_RAMBA_RETRAIN_RUN_ID', 'best_ema_revision_v1')
MODEL_DIR = Path(os.environ.get('ECG_RAMBA_MODEL_DIR', str(MODEL_RUNS_DIR / MODEL_RUN_ID))).expanduser()
LEGACY_MODEL_DIR = DRIVE_ROOT / 'model'
CURRENT_MODEL_POINTER = MODEL_RUNS_DIR / 'current_best_ema_model_dir.txt'

from google.colab import drive
drive.mount('/content/drive')

def run(cmd, *, cwd=None, check=True):
    print('$', cmd)
    return subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=check)

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_RUNS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
if MODEL_DIR.resolve() == LEGACY_MODEL_DIR.resolve():
    raise RuntimeError('Refusing to retrain into the historical Drive/ECG-Ramba/model directory. Set ECG_RAMBA_MODEL_DIR to a model_runs subdirectory.')

if not REPO_DIR.exists():
    run(f'git clone {REPO_URL} "{REPO_DIR}"')
run('git fetch origin', cwd=REPO_DIR)
run(f'git checkout {BRANCH}', cwd=REPO_DIR)
run(f'git pull --ff-only --autostash origin {BRANCH}', cwd=REPO_DIR)
os.chdir(REPO_DIR)

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['ECG_RAMBA_MODEL_DIR'] = str(MODEL_DIR)
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '1')

print('cwd       :', Path.cwd())
print('drive root:', DRIVE_ROOT)
print('model dir :', MODEL_DIR)
print('legacy dir:', LEGACY_MODEL_DIR)
print('pointer   :', CURRENT_MODEL_POINTER)
print('branch    :', subprocess.check_output('git branch --show-current', shell=True, text=True).strip())
print('commit    :', subprocess.check_output('git rev-parse HEAD', shell=True, text=True).strip())


## Runtime Sanity Check


In [ ]:
import importlib.util
import torch

print('Python:', sys.version)
print('Torch :', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU   :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    raise RuntimeError('Retraining requires a CUDA GPU. Select A100 High-RAM before running this notebook.')

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print('GPU RAM:', f'{gpu_mem_gb:.1f} GiB')
if gpu_mem_gb < 35:
    raise RuntimeError('Use A100 High-RAM for full five-fold retraining. T4/L4 can be too slow or memory-limited.')

for module in ['mamba_ssm', 'causal_conv1d']:
    if importlib.util.find_spec(module) is None:
        raise ImportError(f'{module} is missing. Run Notebook 00 bootstrap/runtime install first, then restart and rerun this notebook.')
    print('OK import:', module)


## Data And Cache Preflight


In [ ]:
from configs.config import PATHS, CONFIG_HASH, CONFIG

required = {
    'Chapman ZIP': Path(PATHS['zip_path']),
    'cache dir': Path(PATHS['cache_dir']),
    'model dir': Path(PATHS['model_dir']),
}
for name, path in required.items():
    print(f'{name:12s}: exists={path.exists()} path={path}')
if not Path(PATHS['zip_path']).is_file():
    raise FileNotFoundError(f'Missing Chapman ZIP: {PATHS["zip_path"]}')

print('Config hash:', CONFIG_HASH)
print('Folds      :', CONFIG['n_folds'])
print('Epochs     :', CONFIG['epochs'])
print('Batch size :', CONFIG['batch_size'])
print('Aggregation:', CONFIG['aggregation_method'], 'q=', CONFIG['power_mean_q'])


## Run Five-Fold Retraining


In [ ]:
from datetime import datetime, timezone
import subprocess

RUN_RETRAIN = True
ALLOW_OVERWRITE_MODEL_RUN = os.environ.get('ECG_RAMBA_ALLOW_OVERWRITE_MODEL_RUN', '0') == '1'

log_dir = Path('reports/revision/logs')
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / 'retrain_best_ema_train.log'
cmd = [sys.executable, '-u', 'scripts/train.py']
print('Training log:', log_path)
existing_ckpts = sorted(MODEL_DIR.glob('fold*_*.pt'))
if RUN_RETRAIN and existing_ckpts and not ALLOW_OVERWRITE_MODEL_RUN:
    preview = ', '.join(path.name for path in existing_ckpts[:8])
    raise RuntimeError(
        f'Model run directory already contains {len(existing_ckpts)} checkpoint files: {MODEL_DIR}. '
        f'Preview: {preview}. Set ECG_RAMBA_RETRAIN_RUN_ID to a new run id, set RUN_RETRAIN=False to inspect only, '
        'or set ECG_RAMBA_ALLOW_OVERWRITE_MODEL_RUN=1 to intentionally overwrite this run.'
    )
print('$', ' '.join(cmd))

if RUN_RETRAIN:
    with log_path.open('w', encoding='utf-8') as log:
        log.write('started_utc=' + datetime.now(timezone.utc).isoformat() + '\n')
        log.write('model_dir=' + str(MODEL_DIR) + '\n')
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        returncode = proc.wait()
        log.write('finished_utc=' + datetime.now(timezone.utc).isoformat() + '\n')
        log.write('returncode=' + str(returncode) + '\n')
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)
    print('Retraining finished:', log_path)
else:
    print('RUN_RETRAIN=False; skipping training and leaving existing artifacts untouched.')


## Verify Explicit Checkpoint Contract


In [ ]:
import json
import hashlib
import pandas as pd
import torch

model_dir = Path(os.environ['ECG_RAMBA_MODEL_DIR'])
expected = []
for fold in range(1, 6):
    expected.extend([
        model_dir / f'fold{fold}_best_ema.pt',
        model_dir / f'fold{fold}_best_raw.pt',
        model_dir / f'fold{fold}_final_ema.pt',
        model_dir / f'fold{fold}_final_raw.pt',
        model_dir / f'fold{fold}_best.pt',
        model_dir / f'fold{fold}_final.pt',
    ])
expected.extend([
    model_dir / 'folds.pkl',
    model_dir / 'training_log_epochs.csv',
    model_dir / 'cv_results_clean_core.csv',
])
missing = [str(path) for path in expected if not path.exists()]
if missing:
    raise FileNotFoundError('Missing retrain artifacts: ' + '; '.join(missing))

rows = []
for fold in range(1, 6):
    ckpt = model_dir / f'fold{fold}_best_ema.pt'
    payload = torch.load(ckpt, map_location='cpu')
    if payload.get('weights_kind') != 'ema':
        raise RuntimeError(f'{ckpt} does not report weights_kind=ema')
    if payload.get('selected_by_weights_kind') != 'ema':
        raise RuntimeError(f'{ckpt} was not selected by EMA validation')
    h = hashlib.sha256(ckpt.read_bytes()).hexdigest()
    rows.append({
        'fold': fold,
        'path': str(ckpt),
        'sha256': h,
        'epoch': payload.get('epoch'),
        'f1_macro': payload.get('f1_macro'),
        'weights_kind': payload.get('weights_kind'),
        'selected_by_weights_kind': payload.get('selected_by_weights_kind'),
        'config_hash': payload.get('config_hash'),
        'aggregation': payload.get('aggregation'),
        'pca_explained_variance': payload.get('pca_explained_variance'),
    })

manifest_payload = {
    'model_run_id': MODEL_RUN_ID,
    'model_dir': str(model_dir),
    'legacy_model_dir': str(LEGACY_MODEL_DIR),
    'current_model_pointer': str(CURRENT_MODEL_POINTER),
    'overwrite_legacy_model_dir': False,
    'checkpoints': rows,
}
manifest_path = Path('reports/revision/manifests/retrain_best_ema_checkpoint_manifest.json')
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest_payload, indent=2, sort_keys=True), encoding='utf-8')
CURRENT_MODEL_POINTER.parent.mkdir(parents=True, exist_ok=True)
CURRENT_MODEL_POINTER.write_text(str(model_dir), encoding='utf-8')
print('Wrote:', manifest_path)
print('Wrote current model pointer:', CURRENT_MODEL_POINTER, '->', model_dir)
display(pd.DataFrame(rows))

cv_path = model_dir / 'cv_results_clean_core.csv'
train_log_path = model_dir / 'training_log_epochs.csv'
print('CV results:', cv_path)
display(pd.read_csv(cv_path))
print('Training log tail:')
display(pd.read_csv(train_log_path).tail(10))
